# 4.3 预测房价：标量回归问题示例

前面的影评分类和新闻分类都属于**分类问题**，预测结果是离散的类别标签。

本节学习的是另一种常见问题：**回归问题（Regression）**。

## 什么是回归问题？

回归问题的预测结果不是一个类别，而是一个**连续数值**。

例如：预测房价、气温、销量、考试分数等。

本实验任务：根据波士顿郊区房屋的已知特征，预测该房屋的中位数价格。

---

## 本实验将学习到的内容

1. 什么是标量回归问题
2. 如何加载波士顿房价数据集
3. 为什么回归任务需要对数据进行标准化
4. 如何构建适用于回归问题的神经网络
5. 为什么最后一层不使用激活函数
6. 为什么损失函数用 `mse`
7. 为什么评价指标常用 `mae`
8. 如何使用 K 折交叉验证评估模型

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import boston_housing
from tensorflow.keras import models
from tensorflow.keras import layers

## 1. 波士顿房价数据集简介

Boston Housing 是一个经典的回归数据集。

### 输入特征
每个样本有 13 个数值特征：犯罪率、住宅平均房间数、财产税率、师生比例等。

### 预测目标
目标值是房价中位数，单位为"千美元"。
- 例如目标值 20.0 表示房价约 20000 美元

### 什么是回归问题？

简单说就是**预测一个具体的数字**，而不是判断属于哪个类别。

- 分类问题：这封邮件是不是垃圾邮件？→ 输出是/否
- 回归问题：这套房子值多少钱？→ 输出 23.5（万美元）

只要答案是连续的数值（可以是任意小数），就是回归问题。生活中常见的例子：预测明天气温、预测股价、预测一个人的身高体重……都是回归。

In [2]:
(train_data, train_targets), (test_data, test_targets) = boston_housing.load_data()

print("训练集特征 shape:", train_data.shape)
print("测试集特征 shape:", test_data.shape)
print("训练集标签 shape:", train_targets.shape)
print("测试集标签 shape:", test_targets.shape)

57026/57026 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
训练集特征 shape: (404, 13)
测试集特征 shape: (102, 13)
训练集标签 shape: (404,)
测试集标签 shape: (102,)


In [ ]:
print("第一条训练样本的13个特征值：")
print(train_data[0])

print("\n第一条训练样本对应的房价：")
print(train_targets[0])

第一条训练样本的13个特征值：
[  1.23247   0.        8.14      0.        0.538     6.142    91.7
   3.9769    4.      307.       21.      396.9      18.72   ]

第一条训练样本对应的房价：
15.2


## 2. 准备数据：标准化

不同特征的取值范围差别很大，直接输入神经网络会导致：
1. 模型训练困难
2. 数值范围大的特征可能"主导"训练

### 标准化公式
```
标准化后的值 = (原值 - 均值) / 标准差
```

**注意**：标准化测试集时必须使用**训练集的均值和标准差**，因为测试集相当于"未来数据"。

In [ ]:
# 计算训练集每个特征的均值（axis=0 按列计算,最后也就是只会有一行，每个特征单独算）
# 补充：axis=1 就是沿着特征方向压缩，每行算一个均值，最后也就是维度变成(404,)
mean = train_data.mean(axis=0)
# 减去均值，让每个特征中心化到0附近
train_data -= mean

# 计算训练集每个特征的标准差
std = train_data.std(axis=0)
# 除以标准差，让每个特征的数值范围统一到1左右
train_data /= std

# 测试集用训练集的均值和标准差来归一化，而不是自己重新算
# 因为现实中预测新数据时，你不知道"未来数据"的均值，只能用训练集的
test_data -= mean
test_data /= std

In [ ]:
print("标准化后的第一条训练样本：")
print(train_data[0])

标准化后的第一条训练样本：
[-0.27224633 -0.48361547 -0.43576161 -0.25683275 -0.1652266  -0.1764426
  0.81306188  0.1166983  -0.62624905 -0.59517003  1.14850044  0.44807713
  0.8252202 ]


## 3. 构建回归模型

由于样本数量较少，网络不能太复杂，否则容易过拟合。

### 网络结构
- 输入层：13维
- 隐藏层1：64个神经元，ReLU
- 隐藏层2：64个神经元，ReLU
- 输出层：1个神经元

### 为什么输出层不使用激活函数？
- 分类问题用 sigmoid/softmax 限制输出范围
- 回归问题预测任意连续值，最后一层不加激活函数（**线性输出层**）

### 为什么损失函数用 MSE？
MSE（均方误差）反映预测值与真实值的整体偏差。

### 为什么评价指标用 MAE？
MAE（平均绝对误差）表示预测值与真实值平均相差多少。例如 MAE=2.5 表示平均误差约 2.5 千美元。

> 总结：一句话总结：MSE 适合训练（大误差惩罚更重，梯度更敏感），MAE 适合看结果（单位直观，人能理解）。

In [3]:
def build_model():
    model = models.Sequential()

    # 第一隐藏层：64个神经元，ReLU激活，input_shape指定输入特征维度
    model.add(layers.Dense(64, activation="relu", input_shape=(train_data.shape[1],)))

    # 第二隐藏层：64个神经元，ReLU激活
    model.add(layers.Dense(64, activation="relu"))

    # 输出层：1个神经元，无激活函数（回归任务直接输出实数值）
    model.add(layers.Dense(1))

    model.compile(
        optimizer="rmsprop",  # RMSProp优化器，适合小数据集
        loss="mse",           # 均方误差损失，回归任务常用
        metrics=["mae"]       # 监控平均绝对误差，便于直观评估预测偏差
    )
    return model

## 4. 使用 K 折交叉验证评估模型


数据量较小（训练集仅404条），普通验证集划分会导致结果波动大。

**为什么数据少了会波动大？**

数据只有404条，如果随机切出100条做验证集，很可能这100条碰巧都是"简单样本"或"难样本"，导致评估结果全靠运气——换一种切法结果可能差很多。

**K 折交叉验证思想（K=4）**

把404条数据平均切成4份，每份约101条，然后轮流做验证集：

```
第1次：[验证集] [训练] [训练] [训练] → 得到 MAE₁
第2次：[训练] [验证集] [训练] [训练] → 得到 MAE₂
第3次：[训练] [训练] [验证集] [训练] → 得到 MAE₃
第4次：[训练] [训练] [训练] [验证集] → 得到 MAE₄

最终 MAE = (MAE₁ + MAE₂ + MAE₃ + MAE₄) / 4
```

**好处：** 每份数据都当过一次验证集，最终取平均，评估结果不会因为"切法运气好坏"而失真，比单次划分**稳定得多**。

代价就是要训练4次，耗时是普通验证的4倍。数据越少，K折越值得做。

In [ ]:
k = 4
num_val_samples = len(train_data) // k  # 每折的样本数：404 // 4 = 101
num_epochs = 100
all_scores = []  # 收集每折的 MAE

for i in range(k):
    print(f"正在处理第 {i + 1} 折验证...")

    # 切出第 i 份作为验证集
    val_data    = train_data[i * num_val_samples: (i + 1) * num_val_samples]
    val_targets = train_targets[i * num_val_samples: (i + 1) * num_val_samples]

    # 剩余3份拼接成训练集（第i份前面的 + 第i份后面的）
    partial_train_data = np.concatenate(
        [train_data[:i * num_val_samples],
         train_data[(i + 1) * num_val_samples:]],
        axis=0
    )
    partial_train_targets = np.concatenate(
        [train_targets[:i * num_val_samples],
         train_targets[(i + 1) * num_val_samples:]],
        axis=0
    )

    # 每折重新建模，避免上一折的权重影响这一折
    model = build_model()
    # verbose=0 关掉训练日志，不然100轮×4折输出太多
    model.fit(partial_train_data, partial_train_targets,
              epochs=num_epochs, batch_size=16, verbose=0)

    # 在验证集上评估，取出 MAE
    val_mse, val_mae = model.evaluate(val_data, val_targets, verbose=0)
    all_scores.append(val_mae)

print("每一折的验证 MAE：", all_scores)
print("平均验证 MAE：", np.mean(all_scores))  # 4折平均，结果更稳定

## 5. 更细致地观察：每个 epoch 的验证误差

为了确定训练多少轮最合适，我们记录每个 epoch 的验证 MAE，然后观察过拟合开始的时间点。

In [ ]:
num_epochs = 500  # 训练轮数增加到500，观察更完整的训练曲线
all_mae_histories = []  # 收集每折每轮的验证MAE

for i in range(k):
    print(f"正在处理第 {i + 1} 折详细验证...")

    # 切出验证集（同前）
    val_data    = train_data[i * num_val_samples: (i + 1) * num_val_samples]
    val_targets = train_targets[i * num_val_samples: (i + 1) * num_val_samples]

    # 拼接训练集（同前）
    partial_train_data = np.concatenate(
        [train_data[:i * num_val_samples],
         train_data[(i + 1) * num_val_samples:]],
        axis=0
    )
    partial_train_targets = np.concatenate(
        [train_targets[:i * num_val_samples],
         train_targets[(i + 1) * num_val_samples:]],
        axis=0
    )

    model = build_model()
    history = model.fit(
        partial_train_data, partial_train_targets,
        validation_data=(val_data, val_targets),  # 这次传入验证集，记录每轮MAE
        epochs=num_epochs, batch_size=16, verbose=0
    )

    # 取出这一折500轮的验证MAE列表，长度=500
    mae_history = history.history["val_mae"]
    all_mae_histories.append(mae_history)  # 追加到总列表，最终形状：4×500

In [ ]:
average_mae_history = [
    np.mean([x[i] for x in all_mae_histories]) for i in range(num_epochs)
]

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(average_mae_history) + 1), average_mae_history)
plt.xlabel("Epochs")
plt.ylabel("Validation MAE")
plt.title("Average validation MAE")
plt.grid(True)
plt.show()

## 6. 去掉前几轮再观察

训练刚开始时模型参数是随机的，前几轮验证误差变化剧烈。

去掉前面若干轮后，曲线更平滑，便于观察整体趋势。

In [ ]:
truncated_mae_history = average_mae_history[10:]

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(truncated_mae_history) + 1), truncated_mae_history)
plt.xlabel("Epochs")
plt.ylabel("Validation MAE")
plt.title("Validation MAE (excluding first 10 epochs)")
plt.grid(True)
plt.show()

## 7. 选择最终训练轮数

观察验证 MAE 曲线：
- 前期误差下降
- 某个 epoch 后不再下降
- 继续训练误差可能上升（过拟合）

选择验证误差最低附近的 epoch 数作为最终训练轮数。

In [ ]:
# 根据观察结果，选择训练80轮
model = build_model()
model.fit(train_data, train_targets, epochs=80, batch_size=16, verbose=0)

## 8. 在测试集上评估模型

主要关注：
- `mse`：均方误差
- `mae`：平均绝对误差（更易解释）

例如 MAE=2.5 表示平均误差约 2.5 千美元。

In [ ]:
test_mse_score, test_mae_score = model.evaluate(test_data, test_targets)
print("测试集 MSE：", test_mse_score)
print("测试集 MAE：", test_mae_score)

## 9. 使用模型进行预测

回归模型输出的是具体数值。

In [ ]:
predictions = model.predict(test_data)

print("前5条测试样本的预测房价：")
print(predictions[:5])

In [ ]:
print("前5条测试样本的真实房价：")
print(test_targets[:5])

## 10. 实验总结

本实验完成了**标量回归问题：房价预测**。

### 主要步骤
1. **加载数据**：Boston Housing 数据集
2. **数据标准化**：(x - mean) / std
3. **构建模型**：Dense(64, relu) -> Dense(64, relu) -> Dense(1)
4. **模型编译**：rmsprop + mse + mae
5. **K 折交叉验证**：更稳定的模型评估
6. **选择训练轮数**：观察验证 MAE 曲线避免过拟合
7. **测试评估**：关注 MAE 指标

---

### 回归 vs 分类
| 分类任务 | 回归任务 |
|---------|---------|
| 输出是类别 | 输出是连续数值 |
| 用 sigmoid/softmax | 输出层不加激活函数 |
| 用 crossentropy | 用 mse |

### 关键概念
- **回归**：预测连续数值
- **标准化**：(x - mean) / std
- **MSE**：均方误差
- **MAE**：平均绝对误差
- **K 折交叉验证**：适合小数据集的评估方法